# Chapter 6 - Neural Networks
## 1. Implementation of the Perceptron Training and Testing Algorithms

### 1.1. Implementation of the Perceptron Training Algorithm

This function aims to find the optimal weight vector $w$ (including bias) after a certain number of epochs.

Weight update mechanism:
- Error $E_j$ is calculated: $E_j = y_j - predicted\_y_j$, where $y_j$ is the actual class.

- Weight adjustment: $w_{\text{new}} = w_{\text{old}} + \Delta w$, where $\Delta w = \alpha x_j E_j$. ($\alpha$ is LearningRate).

In [ ]:
import numpy as np

def PerceptronLearning(TrainingSet, Class, Epochs, LearningRate):
    """
    Thực hiện thuật toán huấn luyện Perceptron (Perceptron Learning Algorithm).

    Args:
        TrainingSet (np.array): Ma trận dữ liệu huấn luyện (n instances x m+1 features).
                                CẦN ĐẢM BẢO cột bias (luôn bằng 1) đã được thêm vào cuối ma trận.
        Class (np.array): Vector các lớp thực tế (1 hoặc -1).
        Epochs (int): Số lượng epoch.
        LearningRate (float): Tốc độ học (alpha) nằm trong khoảng (0, 1).

    Returns:
        np.array: Vector trọng số đã được huấn luyện (bao gồm cả bias).
    """

    n, m_plus_1 = TrainingSet.shape  # n: số lượng mẫu, m_plus_1: số lượng features + bias

    # Khởi tạo vector trọng số ngẫu nhiên
    # m_plus_1 là kích thước của vector trọng số (features + bias)
    w = 0.5 * np.random.rand(m_plus_1)

    alpha = LearningRate

    for epoch in range(1, Epochs + 1):
        # Biến cờ để kiểm tra sự hội tụ
        converged = True

        # Lặp qua từng trường hợp (instance)
        for j in range(n):
            x_j = TrainingSet[j, :]  # Vector đặc trưng thứ j (bao gồm bias)
            y_j = Class[j]           # Lớp thực tế

            # Tính Tổng Trọng số: wsum = w * x_j (dot product)
            wsum = np.dot(w, x_j)

            # Áp dụng Hàm Kích hoạt (Sign function): 1 nếu > 0, -1 nếu <= 0
            y_predicted = 1 if wsum > 0 else -1

            # Tính toán Lỗi: Error = Class(j) - y
            Error = y_j - y_predicted

            # Điều chỉnh Trọng số nếu có lỗi (Error != 0)
            if Error != 0:
                # Delta_w = alpha * x_j * Error (Đảo vị trí các thừa số nhưng kết quả là như nhau)
                Delta_w = Error * x_j * alpha
                w = w + Delta_w
                converged = False

        # Dừng sớm nếu không có lỗi nào xảy ra trong epoch này
        if converged:
            break

    return w

### 1.2. Implementation of the Perceptron Testing Algorithm
This function uses the trained weight vector $w$ to predict the class for the test set and calculates the accuracy.

Accuracy calculation mechanism: Accuracy is calculated as the percentage between correctly classified instances and the total number of instances.

$$ \text{Accuracy} = \left( 1 - \frac{incorrect\_predictions}{total\_instances} \right) \times 100 $$

In [ ]:
import numpy as np

def PerceptronTesting(TestingSet, Class, w):
    """
    Thực hiện kiểm tra (testing) cho mạng Perceptron đơn lớp.

    Args:
        TestingSet (np.array): Ma trận dữ liệu kiểm tra (N instances x m+1 features).
                                CẦN ĐẢM BẢO cột bias (luôn bằng 1) đã được thêm vào cuối ma trận.
        Class (np.array): Vector các lớp thực tế tương ứng (1 hoặc -1).
        w (np.array): Vector trọng số đã được huấn luyện (1 x m+1).

    Returns:
        tuple: (PredictedClass, Accuracy)
    """

    N_instances, _ = TestingSet.shape  # Lấy số lượng trường hợp (số hàng)

    # Khởi tạo vector lớp dự đoán với kích thước N_instances
    PredictedClass = np.zeros(N_instances)

    for j in range(N_instances):
        x = TestingSet[j, :]  # Vector đặc trưng x_j

        # Tính Tổng Trọng số: wsum = w * x
        wsum = np.dot(w, x)

        # Áp dụng Hàm Kích hoạt (Sign Function)
        if wsum > 0:
            PredictedClass[j] = 1
        else:
            PredictedClass[j] = -1

    # Tính Độ lỗi (Error): Error = Class - PredictedClass
    Error = Class - PredictedClass

    # Số lượng các lỗi khác 0 (trường hợp phân loại sai)
    incorrect_predictions = np.count_nonzero(Error)

    # Tổng số trường hợp kiểm tra
    total_instances = len(Class)

    # Tính Độ chính xác (Accuracy) theo công thức của nguồn
    Accuracy = (1 - (incorrect_predictions / total_instances)) * 100

    return PredictedClass, Accuracy

### 1.3. Implementation of the Training and Testing Process
We will use a small linearly separable dataset (e.g. logical AND) to illustrate the training and testing process.

In [ ]:
# 1. Chuẩn bị Dữ liệu (Logic AND - Tách biệt tuyến tính)
# Dữ liệu 4x3: (x1, x2, bias=1)
# Kết quả mong muốn: (0,0) -> -1; (0,1) -> -1; (1,0) -> -1; (1,1) -> 1

# TrainingSet phải bao gồm cột bias (luôn bằng 1)
# Giả sử ta đang làm việc với 2 feature x1, x2 và 1 bias (tổng cộng 3 cột)
training_set = np.array([
    [0.0, 0.0, 1.0],
    [0.0, 1.0, 1.0],
    [1.0, 0.0, 1.0],
    [1.0, 1.0, 1.0]
])

# Lớp thực tế (target class)
actual_classes = np.array([-1, -1, -1, 1])

# Tham số huấn luyện
learning_rate = 0.1
epochs = 100

print("--- BẮT ĐẦU HUẤN LUYỆN PERCEPTRON ---")

# 2. HUẤN LUYỆN MÔ HÌNH
trained_weights = PerceptronLearning(training_set, actual_classes, epochs, learning_rate)

print(f"Vector trọng số đã huấn luyện (features + bias): {trained_weights}")

# 3. KIỂM TRA MÔ HÌNH
# Sử dụng chính tập huấn luyện làm tập kiểm tra để đơn giản hóa ví dụ
testing_set = training_set
testing_classes = actual_classes

predicted_classes, accuracy = PerceptronTesting(testing_set, testing_classes, trained_weights)

print("\n--- KẾT QUẢ KIỂM TRA ---")
print(f"Lớp thực tế: {testing_classes}")
print(f"Lớp dự đoán: {predicted_classes}")
print(f"Độ chính xác trên tập kiểm tra: {accuracy:.2f}%")

# 4. KIỂM TRA TRƯỜNG HỢP MỚI (Ví dụ)
new_instance = np.array([1.0, 0.0, 1.0]) # Input [1, 0, bias=1]. Dự đoán: -1
wsum_new = np.dot(trained_weights, new_instance)
prediction_new = 1 if wsum_new > 0 else -1

print(f"\nKiểm tra mẫu mới: Tổng trọng số = {wsum_new:.4f}, Dự đoán = {prediction_new}")


## 2. Implementation of a Neural Network

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import os

# --- BƯỚC 1: Tải và Chuẩn bị Dữ liệu Ecoli từ ecoli.txt ---

FILE_NAME = 'ecoli.txt' # Đã đổi tên file

# Các tham số cấu hình mạng
HIDDEN_LAYER_SIZE = (10,) # 1 lớp ẩn với 10 nơ-ron
# Tương đương với Sigmoid function S1(x) = 1/(1 + e^-x)
ACTIVATION_FUNCTION = 'logistic'
MAX_EPOCHS = 300
RANDOM_STATE = 42 #random seed

if not os.path.exists(FILE_NAME):
    print(f"Lỗi: Không tìm thấy tệp tin '{FILE_NAME}' trong thư mục hiện hành.")
    print("Vui lòng đảm bảo rằng tệp 'ecoli.txt' đã được đặt đúng chỗ.")
    # Khởi tạo dữ liệu giả định nếu file không tồn tại (chỉ để code không bị crash)
    X = np.random.rand(336, 7) # Ecoli thường có 7 features
    y = np.random.randint(0, 8, 336) # Ecoli thường có 8 classes
else:
    print(f"Đang tải dữ liệu từ '{FILE_NAME}'...")
    try:
        # Đọc file TXT sử dụng dấu chấm phẩy làm delimiter
        data = pd.read_csv(FILE_NAME, header=None, sep=';')

        # Tách Features (X) và Target (y)
        # BỎ qua cột 0 (định danh: AAT_ECOLI)
        # X: các cột từ 1 đến hết-trừ-1 (features)
        X = data.iloc[:, 1:-1].values
        # y: cột cuối cùng (classes)
        y = data.iloc[:, -1].values

        # Mã hóa nhãn lớp (Label Encoding): Cần thiết vì nhãn lớp là chuỗi ký tự (cp, im, om...)
        le = LabelEncoder()
        y = le.fit_transform(y)

        print(f"Đã tải {X.shape} mẫu với {X.shape} thuộc tính.")

    except Exception as e:
        print(f"Lỗi khi đọc file TXT: {e}")
        X = np.random.rand(300, 7)
        y = np.random.randint(0, 8, 300)


# --- BƯỚC 3: Chia Dữ liệu (Tỷ lệ 70% Train, 20% Validation, 10% Test) ---

# Lần 1: Tách 90% (Train + Val) và 10% Test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.1, random_state=RANDOM_STATE, stratify=y
)

# Lần 2: Tách Train và Validation từ 90% còn lại.
# Tỷ lệ validation trên tập X_train_val là (0.2 / 0.9) = 0.22
val_ratio_of_remaining = 0.2 / 0.9
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=val_ratio_of_remaining,
    random_state=RANDOM_STATE, stratify=y_train_val
)

print(f"\nKích thước tập huấn luyện (70%): {X_train.shape}")
print(f"Kích thước tập xác thực (20%): {X_val.shape}")
print(f"Kích thước tập kiểm tra (10%): {X_test.shape}")


# --- BƯỚC 4: Khởi tạo và Huấn luyện Mạng ---

# Thiết lập MLPClassifier
# hidden_layer_sizes=(10,): 1 lớp ẩn với 10 nơ-ron
# activation='logistic': Hàm Sigmoid
# early_stopping=True: Mô phỏng việc sử dụng tập xác thực (Validation) để dừng sớm
mlp = MLPClassifier(
    hidden_layer_sizes=HIDDEN_LAYER_SIZE,
    activation=ACTIVATION_FUNCTION,
    max_iter=MAX_EPOCHS,
    random_state=RANDOM_STATE,
    early_stopping=True,
    # Validation_fraction cần bằng với tỷ lệ X_val/X_train_val
    validation_fraction=val_ratio_of_remaining,
    n_iter_no_change=20
)

print("\nBắt đầu huấn luyện...")
mlp.fit(X_train, y_train)
print(f"Huấn luyện hoàn tất sau {mlp.n_iter_} epoch.")


# --- BƯỚC 5: ĐÁNH GIÁ HIỆU SUẤT (Performance) ---

# Đánh giá trên tập Kiểm tra (Test)
y_test_pred = mlp.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)

# Performance (Loss/Error)
loss = mlp.loss_

print("\n--- KẾT QUẢ ĐÁNH GIÁ (Performance) ---")
# Loss trong Scikit-learn là Cross-Entropy Loss.
print(f"Total Loss trên tập huấn luyện: {loss:.4f}")
print(f"Độ chính xác trên tập Kiểm tra (Test): {test_accuracy * 100:.2f}%")

**Các tham số Thiết lập MLPClassifier**

| Tham số                                          | Ý nghĩa                                                                                                                                                                                                                                                                  |
| ------------------------------------------------ | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| **`hidden_layer_sizes=HIDDEN_LAYER`**    | Xác định **số lượng nơ-ron trong các lớp ẩn**. <br> - Ví dụ: `(100,)` nghĩa là có **1 lớp ẩn với 100 nơ-ron**.<br> - Có thể mở rộng, ví dụ `(64, 32, 16)` → 3 lớp ẩn.                                   |
| **`activation=ACTIVATION_FUNCTION`**             | Hàm kích hoạt (activation function) dùng cho các nơ-ron trong lớp ẩn. <br> Các giá trị thường gặp: <br> - `'relu'`: Rectified Linear Unit (mặc định, phổ biến nhất) <br> - `'tanh'`: hyperbolic tangent <br> - `'logistic'`: sigmoid <br> - `'identity'`: hàm tuyến tính |
| **`max_iter=MAX_EPOCHS`**                        | Số vòng lặp tối đa (epochs) cho quá trình huấn luyện. <br> Sau số lần lặp này, nếu mô hình chưa hội tụ, quá trình dừng lại.                                                                                                                                              |
| **`random_state=RANDOM_STATE`**                  | Thiết lập **hạt giống ngẫu nhiên** để đảm bảo kết quả huấn luyện **có thể tái lặp** (deterministic).                                                                                                                                                                     |
| **`early_stopping=True`**                        | Bật **cơ chế dừng sớm**: <br> - Trong quá trình huấn luyện, một phần dữ liệu sẽ được tách ra làm **tập validation**. <br> - Nếu mô hình không cải thiện trong một số vòng liên tiếp (xác định bởi `n_iter_no_change`), quá trình sẽ **tự dừng** để tránh overfitting.    |
| **`validation_fraction=val_ratio_of_remaining`** | Tỉ lệ dữ liệu được dùng làm **tập validation** khi `early_stopping=True`. <br> Ví dụ: `0.1` nghĩa là dùng **10% dữ liệu huấn luyện** để kiểm định sớm.                                                                                                                   |
| **`n_iter_no_change=20`**                        | Nếu sau **20 vòng liên tiếp** không thấy cải thiện trên tập validation, mô hình sẽ **tự động dừng sớm**.                                                                                                                                                           